In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

train = pd.read_csv('train_sequences.csv')
test = pd.read_csv('test_sequences.csv')

x_train = train.drop(columns='y').values
y_train = train['y'].values
x_test = test.values

a = x_train[:, [0, 3, 6, 9, 12, 15, 18, 21]]
b = x_train[:, [1, 4, 7, 10, 13, 16, 19, 22]]
c = x_train[:, [2, 5, 8, 11, 14, 17, 20, 23]]
codons_train = 16 * a + 4 * b + c

counts_train = np.zeros((800, 64))

for k in range(64):
    counts_train[:, k] = (codons_train == k).sum(axis=1)

model = LinearRegression()
model.fit(counts_train, y_train)


a_ = x_test[:, [0, 3, 6, 9, 12, 15, 18, 21]]
b_ = x_test[:, [1, 4, 7, 10, 13, 16, 19, 22]]
c_ = x_test[:, [2, 5, 8, 11, 14, 17, 20, 23]]
codons_test = 16 * a_ + 4 * b_ + c_

counts_test = np.zeros((3000, 64))

for k in range(64):
    counts_test[:, k] = (codons_test == k).sum(axis=1)

pred_test = model.predict(counts_test)

answers = test.copy()
answers['y'] = pred_test
answers.to_csv('answers.csv', index=False)

In [ ]:
!pip install tabpfn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.0/850.0 kB 25.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 27.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [tabpfn]


In [ ]:
from sklearn.metrics import root_mean_squared_error

# делим 800 строк на 640 для обучения и 160 для проверки
idx_tr, idx_val = train_test_split(np.arange(800), test_size=0.2, random_state=0)

# твоя модель: линейная регрессия на 64 кодонах
lin = LinearRegression()
lin.fit(counts_train[idx_tr], y_train[idx_tr])
pred_lin = lin.predict(counts_train[idx_val])
print('Линейная, кодоны:', root_mean_squared_error(y_train[idx_val], pred_lin))

Линейная, кодоны: 0.15647400473256304


In [ ]:
import os
from google.colab import userdata

os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")

In [ ]:
import time
from tabpfn import TabPFNRegressor

def run(X, name, **params):
    start = time.time()
    m = TabPFNRegressor(random_state=0, **params)
    m.fit(X[idx_tr], y_train[idx_tr])
    pred = m.predict(X[idx_val])
    sec = time.time() - start
    print(f'{name}: RMSE = {root_mean_squared_error(y_train[idx_val], pred):.3f}, время = {sec:.1f} с')

run(x_train, 'TabPFN, сырые нуклеотиды')
run(x_train, 'TabPFN, сырые нуклеотиды как категории',
    categorical_features_indices=list(range(24)))
run(counts_train, 'TabPFN, кодоны')

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

TabPFN, сырые нуклеотиды: RMSE = 2.290, время = 20.0 с
TabPFN, сырые нуклеотиды как категории: RMSE = 2.306, время = 1.5 с
TabPFN, кодоны: RMSE = 0.406, время = 1.9 с


In [ ]:
results = {'среднее': [], 'линейная, кодоны': [], 'TabPFN, кодоны': [], 'TabPFN, сырые': []}

for seed in range(5):
    tr, val = train_test_split(np.arange(800), test_size=0.2, random_state=seed)
    yt, yv = y_train[tr], y_train[val]

    results['среднее'].append(root_mean_squared_error(yv, np.full(len(yv), yt.mean())))

    lin = LinearRegression().fit(counts_train[tr], yt)
    results['линейная, кодоны'].append(root_mean_squared_error(yv, lin.predict(counts_train[val])))

    m = TabPFNRegressor(random_state=0).fit(counts_train[tr], yt)
    results['TabPFN, кодоны'].append(root_mean_squared_error(yv, m.predict(counts_train[val])))

    m = TabPFNRegressor(random_state=0).fit(x_train[tr], yt)
    results['TabPFN, сырые'].append(root_mean_squared_error(yv, m.predict(x_train[val])))

for name, r in results.items():
    print(f'{name}: RMSE = {np.mean(r):.3f} ± {np.std(r):.3f}')

среднее: RMSE = 2.256 ± 0.104
линейная, кодоны: RMSE = 0.183 ± 0.018
TabPFN, кодоны: RMSE = 0.455 ± 0.050
TabPFN, сырые: RMSE = 2.135 ± 0.119


In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(24)           # случайный порядок столбцов 0..23
x_shuffled = x_train[:, perm]        # та же таблица, но столбцы перемешаны

m = TabPFNRegressor(random_state=0).fit(x_shuffled[idx_tr], y_train[idx_tr])
print('TabPFN, сырые, столбцы перемешаны:',
      root_mean_squared_error(y_train[idx_val], m.predict(x_shuffled[idx_val])))

TabPFN, сырые, столбцы перемешаны: 2.2900356027857143
